In [1]:
from pyspark.sql import SparkSession

In [2]:

spark = SparkSession.builder \
    .appName("AverageProcessingTime") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/16 18:10:34 WARN Utils: Your hostname, ast-ubuntu, resolves to a loopback address: 127.0.1.1; using 192.168.1.10 instead (on interface wlp2s0)
26/04/16 18:10:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 18:10:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print(spark)

In [7]:
from pyspark.sql.functions import col

path = "/home/akshay/Documents/Setups/1Projects/RetailAnalysis/misc/PySpark/data/test1nb.csv"

input_df = spark.read \
		.option("header", "false") \
		.option("inferSchema", "true") \
		.format('csv') \
		.load(path)
 

input_df = input_df.toDF('Name', 'Age', 'City')


output_df = input_df.filter(col('Age') >= 18)

output_df.show()

+----+---+--------+
|Name|Age|    City|
+----+---+--------+
|  AR| 29|Kolapur |
|  MC| 28| Satara |
|  DK| 23|   Pune |
+----+---+--------+



In [ ]:
# Define schema using StructType instead of inferring
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define the schema explicitly
# StructType assigns column names, so header=false works perfectly with it
schema = StructType([
    StructField("Name", StringType(), True),      # 1st column -> Name (String type)
    StructField("Age", IntegerType(), True),      # 2nd column -> Age (Integer type)
    StructField("City", StringType(), True)       # 3rd column -> City (String type)
])

# Read CSV with explicit schema
# header=false: First row is data, not column names
# schema=schema: Column names and types come from the schema definition
input_df = spark.read \
    .option("header", "false") \
    .schema(schema) \
    .format('csv') \
    .load(path)

output_df = input_df.filter(col('Age') >= 18)
output_df.show()

In [14]:

#output:
#eid, name, manager_name

data = [
    (1, "abc", 6),
    (2, "xyz", 6),
    (3, "agt", 7),
    (4, "agy", 7),
    (5, "ahd", 6),
    (6, "mag", 8),
    (7, "gen", 8),
    (8, "hdg", None),
]

emp_df = spark.createDataFrame(data, ["eid","name","manager_id"])
 
emp_e = emp_df.alias("emp_e")
emp_m = emp_df.alias("emp_m")

 
output_df = emp_e.join(emp_m,col("emp_m.eid") == col("emp_e.manager_id"),"left") \
                    .select(
                        col("emp_e.eid").alias("eid"),
                        col("emp_e.name").alias("name"),
                        col("emp_m.name").alias("manager_name")
			        )
 
output_df.show()
 

+---+----+------------+
|eid|name|manager_name|
+---+----+------------+
|  1| abc|         mag|
|  2| xyz|         mag|
|  3| agt|         gen|
|  4| agy|         gen|
|  5| ahd|         mag|
|  6| mag|         hdg|
|  8| hdg|        NULL|
|  7| gen|         hdg|
+---+----+------------+



In [25]:
"""
input:
eid, name, salary 
1, abc, 100
2, xyz, 200
3, agt, 200
4, agy, 500 
5, ahd, 600
 
output: 
eid, name, salary, cummulative
1, abc, 100, 100
2, xyz, 200, 300
3, agt, 200, 500
4, agy, 500, 1000
5, ahd, 600, 1600
"""

data = [
    (1, "abc", 100),
    (2, "xyz", 200),
    (3, "agt", 300),
    (4, "agy", 400),
    (5, "ahd", 500)
]

input_df = spark.createDataFrame(data, ["eid","name","salary"]) 

from pyspark.sql.window import Window
from pyspark.sql import functions as F
 
window_spec = Window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
 
output_df = input_df.withColumn("cummulative", F.sum(col("salary")).over(window_spec))

output_df.show()

26/04/16 18:38:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 18:38:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 18:38:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 18:38:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 18:38:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+----+------+-----------+
|eid|name|salary|cummulative|
+---+----+------+-----------+
|  1| abc|   100|        100|
|  2| xyz|   200|        300|
|  3| agt|   300|        600|
|  4| agy|   400|       1000|
|  5| ahd|   500|       1500|
+---+----+------+-----------+



In [26]:
spark.stop()